## Assignment 1:

- Using Numpy to implement the soft-margin SVM model. 
- Train this model using SGD method on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.

In [12]:
import numpy as np
import os
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score

# =====================
# SVM IMPLEMENTATION (Soft-Margin SGD)
# =====================
class SVM:
    def __init__(self, lr=0.001, lambda_param=0.01, n_iters=100, batch_size=128):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.batch_size = batch_size
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Label: {0, 1} -> {-1, 1}
        y_ = np.where(y <= 0, -1, 1)

        limit = np.sqrt(1 / n_features)
        self.w = np.random.uniform(-limit, limit, n_features)
        self.b = 0

        sample_weights = np.where(y_ == -1, 2.0, 1.5)

        for epoch in range(self.n_iters):
            # Xáo trộn dữ liệu mỗi epoch
            indices = np.random.permutation(n_samples)
            X_sh = X[indices]
            y_sh = y_[indices]
            sw_sh = sample_weights[indices]

            # Learning rate decay đơn giản
            curr_lr = self.lr / (1 + epoch * 0.01)

            for i in range(0, n_samples, self.batch_size):
                X_batch = X_sh[i : i + self.batch_size]
                y_batch = y_sh[i : i + self.batch_size]
                sw_batch = sw_sh[i : i + self.batch_size]

                # Tính khoảng cách tới lề: y * (Xw + b)
                condition = y_batch * (np.dot(X_batch, self.w) + self.b) < 1
                
                # Chỉ những điểm vi phạm (condition == True) mới đóng góp vào Hinge Loss gradient
                # Thêm trọng số sw_batch vào gradient
                mask = condition.astype(float) * y_batch * sw_batch
                
                dw = (2 * self.lambda_param * self.w) - (np.dot(mask, X_batch) / self.batch_size)
                db = -np.sum(mask) / self.batch_size

                self.w -= curr_lr * dw
                self.b -= curr_lr * db

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{self.n_iters} hoàn thành...")

    def predict(self, X):
        linear_output = np.dot(X, self.w) + self.b
        return np.where(linear_output >= 0, 1, 0)

# =====================
# DATA LOADING & PREPROCESSING
# =====================
def load_and_preprocess(data_dir, img_size=128):
    X, y = [], []
    categories = ['NORMAL', 'PNEUMONIA']
    
    for i, category in enumerate(categories):
        path = os.path.join(data_dir, category)
        if not os.path.exists(path): continue
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (img_size, img_size))
                X.append(img.flatten())
                y.append(i)
            except:
                continue
    
    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    
    # CHUẨN HÓA:
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0) + 1e-7
    X = (X - mean) / std
    
    return X, y

# =====================
# MAIN
# =====================
if __name__ == "__main__":
    TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"
    TEST_DIR = "chest-xray-pneumonia/chest_xray/test"

    print("Đang tải và chuẩn hóa dữ liệu...")
    X_train, y_train = load_and_preprocess(TRAIN_DIR)
    X_test, y_test = load_and_preprocess(TEST_DIR)


    model = SVM(lr=0.0005, lambda_param=0.5, n_iters=100, batch_size=128)
    
    print("Bắt đầu huấn luyện...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("\n" + "="*30)
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
    print("="*30)

Đang tải và chuẩn hóa dữ liệu...
Bắt đầu huấn luyện...
Epoch 10/100 hoàn thành...
Epoch 20/100 hoàn thành...
Epoch 30/100 hoàn thành...
Epoch 40/100 hoàn thành...
Epoch 50/100 hoàn thành...
Epoch 60/100 hoàn thành...
Epoch 70/100 hoàn thành...
Epoch 80/100 hoàn thành...
Epoch 90/100 hoàn thành...
Epoch 100/100 hoàn thành...

Precision: 0.8244
Recall:    0.9026
F1 Score:  0.8617


## Assigment 2:
- Implement SVM method using a machine learning library (such as sklearn or sktorch).
- Train this model on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.
- Compare the results of SVM using library to those of implemented SVM.

In [5]:
import os
import cv2
import numpy as np
from sklearn import svm
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from tqdm import tqdm

def load_and_preprocess(data_dir, img_size=128):
    X, y = [], []
    categories = ['NORMAL', 'PNEUMONIA']
    
    for i, category in enumerate(categories):
        path = os.path.join(data_dir, category)
        if not os.path.exists(path):
            continue

        for img_name in os.listdir(path):
            if img_name.startswith('.'):  # skip hidden files
                continue
            try:
                img_path = os.path.join(path, img_name)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (img_size, img_size))
                X.append(img.flatten())
                y.append(i)
            except:
                continue
    
    X = np.array(X, dtype=np.float32)
    y = np.array(y)

    # Standardization (important for SVM convergence)
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0) + 1e-7
    X = (X - mean) / std
    
    return X, y

TRAIN_DIR = "chest-xray-pneumonia/chest_xray/train"
TEST_DIR = "chest-xray-pneumonia/chest_xray/test"

print("Loading data...")
X_train, y_train = load_and_preprocess(TRAIN_DIR)
X_test, y_test = load_and_preprocess(TEST_DIR)

Loading data...


In [6]:
print("--- Training Sklearn SVM ---")
model_sklearn = svm.SVC(kernel='linear', C=1.0) # Linear nhanh hơn cho dữ liệu lớn
model_sklearn.fit(X_train, y_train)

y_pred_sklearn = model_sklearn.predict(X_test)

--- Training Sklearn SVM ---


In [ ]:
class SimpleSVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=100):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        # Chuyển nhãn 0 thành -1 để phù hợp với toán học SVM
        y_ = np.where(y <= 0, -1, 1)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in tqdm(range(self.n_iters), desc="Training Manual SVM"):
            for idx, x_i in enumerate(X):
                condition = y_[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_[idx]))
                    self.b -= self.lr * y_[idx]

    def predict(self, X):
        approx = np.dot(X, self.w) - self.b
        return np.where(np.sign(approx) <= -1, 0, 1)

model_manual = SimpleSVM(n_iters=10) # Để số vòng lặp thấp vì dữ liệu rất nặng
model_manual.fit(X_train, y_train)
y_pred_manual = model_manual.predict(X_test)

Training Manual SVM: 100%|██████████| 10/10 [00:07<00:00,  1.30it/s]


In [9]:
def evaluate(y_true, y_pred, name):
    print(f"\nResults for {name}:")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")

evaluate(y_test, y_pred_sklearn, "Sklearn SVM")
evaluate(y_test, y_pred_manual, "Manual SVM (SGD)")


Results for Sklearn SVM:
Precision: 0.7520
Recall:    0.9872
F1 Score:  0.8537

Results for Manual SVM (SGD):
Precision: 0.7933
Recall:    0.8462
F1 Score:  0.8189


- Đặc trưng hình ảnh (Feature Complexity): Ảnh X-quang phổi bị viêm và phổi thường rất giống nhau về cấu trúc đại thể. SVM học trên từng Pixel (pixel-wise), nó không hiểu được các cấu trúc hình học (khối mờ, tổn thương kẽ) như CNN.

- Sự dịch chuyển pixel: Chỉ cần bệnh nhân hơi nghiêng người khi chụp, các pixel sẽ bị lệch vị trí. SVM không có tính "bất biến với phép tịnh tiến" (Translation Invariance) nên sẽ bị bối rối.

- Dữ liệu nhiễu: Ảnh X-quang có nhiều chi tiết thừa như xương sườn, bóng tim. SVM nỗ lực tìm một đường thẳng để chia cắt 16,384 chiều, nhưng thực tế ranh giới giữa "bệnh" và "không bệnh" trong y tế là một đường cong cực kỳ phức tạp.